In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================
# Etapa 1: Ingesta — cargar sin limpiar todavía, solo mirar
# ============================================
transaction_data = pd.read_csv("data/transaction_data.csv")
product = pd.read_csv("data/product.csv")
hh_demographic = pd.read_csv("data/hh_demographic.csv")
campaign_table = pd.read_csv("data/campaign_table.csv")
campaign_desc = pd.read_csv("data/campaign_desc.csv")
coupon = pd.read_csv("data/coupon.csv")
coupon_redempt = pd.read_csv("data/coupon_redempt.csv")

# causal_data.csv queda fuera por ahora (no es obligatorio, y es pesado)

# ============================================
# Perfilado de transaction_data
# ============================================

# 1. Estructura general: filas, columnas y tipos de datos
print("=== INFO DE TRANSACTION_DATA ===")
transaction_data.info(show_counts=True)

# 2. Conteo y porcentaje de valores nulos por columna
print("\n=== NULOS POR COLUMNA ===")
null_summary = pd.DataFrame(
    {
        "Nulos": transaction_data.isnull().sum(),
        "Porcentaje (%)": (
            transaction_data.isnull().mean() * 100
        ).round(2),
    }
)
print(null_summary)

# 3. Duplicados — no existe una llave natural que garantice unicidad a nivel de línea:
#    BASKET_ID + PRODUCT_ID puede repetirse legítimamente si el mismo producto fue
#    escaneado por separado dos veces en la misma canasta. .duplicated() solo detecta
#    filas 100% idénticas, que pueden ser reescaneos legítimos, no errores a eliminar.
print("\n=== DUPLICADOS (fila completa idéntica) ===")
print(f"Filas duplicadas: {transaction_data.duplicated().sum():,}")

=== INFO DE TRANSACTION_DATA ===
<class 'pandas.DataFrame'>
RangeIndex: 2595732 entries, 0 to 2595731
Data columns (total 12 columns):
 #   Column             Non-Null Count    Dtype  
---  ------             --------------    -----  
 0   household_key      2595732 non-null  int64  
 1   BASKET_ID          2595732 non-null  int64  
 2   DAY                2595732 non-null  int64  
 3   PRODUCT_ID         2595732 non-null  int64  
 4   QUANTITY           2595732 non-null  int64  
 5   SALES_VALUE        2595732 non-null  float64
 6   STORE_ID           2595732 non-null  int64  
 7   RETAIL_DISC        2595732 non-null  float64
 8   TRANS_TIME         2595732 non-null  int64  
 9   WEEK_NO            2595732 non-null  int64  
 10  COUPON_DISC        2595732 non-null  float64
 11  COUPON_MATCH_DISC  2595732 non-null  float64
dtypes: float64(4), int64(8)
memory usage: 237.6 MB

=== NULOS POR COLUMNA ===
                   Nulos  Porcentaje (%)
household_key          0             0.0
BASK

In [4]:
# 4. Rangos de las numéricas — buscando mínimos negativos y máximos absurdos
print("=== DESCRIBE DE TRANSACTION_DATA ===")
transaction_data.describe()

=== DESCRIBE DE TRANSACTION_DATA ===


,household_key,BASKET_ID,DAY,PRODUCT_ID,QUANTITY,SALES_VALUE,STORE_ID,RETAIL_DISC,TRANS_TIME,WEEK_NO,COUPON_DISC,COUPON_MATCH_DISC
count,2.595732e+06,2.595732e+06,2.595732e+06,2.595732e+06,2.595732e+06,2.595732e+06,2.595732e+06,2.595732e+06,2.595732e+06,2.595732e+06,2.595732e+06,2.595732e+06
mean,1.271953e+03,3.402620e+10,3.887562e+02,2.891435e+06,1.004286e+02,3.104120e+00,3.142673e+03,-5.387054e-01,1.561586e+03,5.622150e+01,-1.641600e-02,-2.918564e-03
std,7.260660e+02,4.711649e+09,1.897210e+02,3.837404e+06,1.153436e+03,4.182274e+00,8.937113e+03,1.249191e+00,3.998378e+02,2.710223e+01,2.168410e-01,3.969004e-02
min,1.000000e+00,2.698485e+10,1.000000e+00,2.567100e+04,0.000000e+00,0.000000e+00,1.000000e+00,-1.800000e+02,0.000000e+00,1.000000e+00,-5.593000e+01,-7.700000e+00
25%,6.560000e+02,3.040805e+10,2.290000e+02,9.174590e+05,1.000000e+00,1.290000e+00,3.300000e+02,-6.900000e-01,1.308000e+03,3.300000e+01,0.000000e+00,0.000000e+00
50%,1.272000e+03,3.276081e+10,3.900000e+02,1.028816e+06,1.000000e+00,2.000000e+00,3.720000e+02,-1.000000e-02,1.613000e+03,5.600000e+01,0.000000e+00,0.000000e+00
75%,1.913000e+03,4.012685e+10,5.530000e+02,1.133018e+06,1.000000e+00,3.490000e+00,4.220000e+02,0.000000e+00,1.843000e+03,8.000000e+01,0.000000e+00,0.000000e+00
max,2.500000e+03,4.230536e+10,7.110000e+02,1.831630e+07,8.963800e+04,8.400000e+02,3.428000e+04,3.990000e+00,2.359000e+03,1.020000e+02,0.000000e+00,0.000000e+00


In [5]:
# 5. Investigando QUANTITY = 0 — ¿qué pasa con SALES_VALUE en esas filas?
print("=== FILAS CON QUANTITY = 0 ===")
quantity_cero = transaction_data[transaction_data["QUANTITY"] == 0]
print(f"Cantidad de filas con QUANTITY = 0: {len(quantity_cero):,}")
print(f"\nDescribe de SALES_VALUE en esas filas:")
print(quantity_cero["SALES_VALUE"].describe())

# Vista rápida de algunos ejemplos para inspeccionar a mano
print("\n=== EJEMPLOS ===")
quantity_cero[["household_key", "BASKET_ID", "PRODUCT_ID", "QUANTITY", "SALES_VALUE", "RETAIL_DISC"]].head(10)

=== FILAS CON QUANTITY = 0 ===
Cantidad de filas con QUANTITY = 0: 14,466

Describe de SALES_VALUE en esas filas:
count    14466.000000
mean         0.001333
std          0.059967
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          5.820000
Name: SALES_VALUE, dtype: float64

=== EJEMPLOS ===


,household_key,BASKET_ID,PRODUCT_ID,QUANTITY,SALES_VALUE,RETAIL_DISC
97,744,26985165432,5978648,0,0.0,0.0
128,1287,26985336468,5978648,0,0.0,0.0
249,2305,26996870743,5978656,0,0.0,0.0
293,271,26997082949,5978656,0,0.0,0.0
694,315,27008952267,957951,0,0.0,0.0
1089,14,27021203242,6463658,0,0.0,0.0
1649,2305,27031060487,5978648,0,0.0,0.0
1985,100,27045626628,5978648,0,0.0,0.0
2142,907,27045792138,1125278,0,0.0,0.0
2143,907,27045792138,5978656,0,0.0,0.0


In [6]:
# 6. Investigando el máximo de QUANTITY (89,638) — ¿un outlier aislado o un patrón?
print("=== TOP 10 TRANSACCIONES CON MAYOR QUANTITY ===")
transaction_data.nlargest(10, "QUANTITY")[["household_key", "BASKET_ID", "PRODUCT_ID", "QUANTITY", "SALES_VALUE"]]

=== TOP 10 TRANSACCIONES CON MAYOR QUANTITY ===


,household_key,BASKET_ID,PRODUCT_ID,QUANTITY,SALES_VALUE
1750942,630,34749153595,6534178,89638,250.00
468356,2407,29392047893,6544236,85055,210.00
481876,630,29484790880,6534178,61335,150.21
166536,149,28210551971,6534178,51912,110.00
1340882,193,32956767959,6534178,48073,121.10
2472791,2133,41904760458,6534178,45475,100.00
1560340,149,33768630428,6534178,41833,115.00
376686,149,29035716247,6534178,41686,100.00
156049,1406,28167562655,6544236,41485,85.00
87625,107,27865225627,6534178,39365,72.00


In [7]:
# 7. ¿Qué son los productos con QUANTITY extremo? Cruzar con product
top_quantity = transaction_data.nlargest(10, "QUANTITY")
top_product_ids = top_quantity["PRODUCT_ID"].unique()

print("=== PRODUCT_IDs detrás de las QUANTITY más altas ===")
product[product["PRODUCT_ID"].isin(top_product_ids)][
    ["PRODUCT_ID", "DEPARTMENT", "COMMODITY_DESC", "SUB_COMMODITY_DESC", "CURR_SIZE_OF_PRODUCT"]
]

=== PRODUCT_IDs detrás de las QUANTITY más altas ===


,PRODUCT_ID,DEPARTMENT,COMMODITY_DESC,SUB_COMMODITY_DESC,CURR_SIZE_OF_PRODUCT
57221,6534178,KIOSK-GAS,COUPON/MISC ITEMS,GASOLINE-REG UNLEADED,
57335,6544236,MISC SALES TRAN,COUPON/MISC ITEMS,GASOLINE-REG UNLEADED,


In [8]:
# 8. ¿Qué son los productos con QUANTITY = 0? Cruzar con product
product_ids_qty_cero = quantity_cero["PRODUCT_ID"].unique()

print(f"Productos distintos involucrados: {len(product_ids_qty_cero)}")
print("\n=== Categorías más frecuentes entre estos productos ===")
productos_qty_cero = product[product["PRODUCT_ID"].isin(product_ids_qty_cero)]
print(productos_qty_cero["COMMODITY_DESC"].value_counts().head(15))

Productos distintos involucrados: 4244

=== Categorías más frecuentes entre estos productos ===
COMMODITY_DESC
SOFT DRINKS                     182
CANDY - PACKAGED                128
CANDY - CHECKLANE                92
BAG SNACKS                       87
BAKED BREAD/BUNS/ROLLS           64
FRZN MEAT/MEAT DINNERS           64
MAKEUP AND TREATMENT             63
COLD CEREAL                      60
FROZEN PIZZA                     58
CANNED JUICES                    57
STATIONERY & SCHOOL SUPPLIES     55
BAKED SWEET GOODS                54
BEEF                             52
LUNCHMEAT                        46
CAKES                            46
Name: count, dtype: int64


In [9]:
# Ver nombres exactos de columnas
print("Columnas en transaction_data:")
print(transaction_data.columns.tolist())

print("\nColumnas en hh_demographic:")
print(hh_demographic.columns.tolist())

Columnas en transaction_data:
['household_key', 'BASKET_ID', 'DAY', 'PRODUCT_ID', 'QUANTITY', 'SALES_VALUE', 'STORE_ID', 'RETAIL_DISC', 'TRANS_TIME', 'WEEK_NO', 'COUPON_DISC', 'COUPON_MATCH_DISC']

Columnas en hh_demographic:
['classification_1', 'classification_2', 'classification_3', 'HOMEOWNER_DESC', 'classification_5', 'classification_4', 'KID_CATEGORY_DESC', 'household_key']


In [10]:
# 9. Obtener los hogares únicos en cada tabla
total_hogares = transaction_data["household_key"].nunique()
hogares_con_demo = hh_demographic["household_key"].nunique()

# 10. Calcular el porcentaje de cobertura
pct_cobertura = (hogares_con_demo / total_hogares) * 100

print("=== COBERTURA DEMOGRÁFICA ===")
print(f"Total hogares en transacciones: {total_hogares:,}")
print(f"Hogares con datos demográficos: {hogares_con_demo:,}")
print(f"Porcentaje de cobertura: {pct_cobertura:.2f}%")
print(f"Hogares sin demografía (NaN): {total_hogares - hogares_con_demo:,}")

=== COBERTURA DEMOGRÁFICA ===
Total hogares en transacciones: 2,500
Hogares con datos demográficos: 801
Porcentaje de cobertura: 32.04%
Hogares sin demografía (NaN): 1,699


### Afirmaciones sobre la calidad de los datos

**1. Cobertura parcial de `hh_demographic` introduce sesgo de selección**

La tabla `hh_demographic` cubre únicamente a 801 de los 2,500 hogares (32.04% de 
cobertura), por lo que cualquier segmentación por ingreso o edad excluye al 68% de 
la base de clientes, introduciendo un sesgo de selección que impide generalizar los 
hallazgos a toda la cadena sin antes validar el comportamiento de compra del segmento 
no identificado.

*(Ver celdas de código 9-10)*

---

**2. `QUANTITY = 0` representa un riesgo de división por cero en cálculos de precio unitario**

La tabla `transaction_data` contiene 14,466 filas con `QUANTITY = 0` (0.56% del total) 
distribuidas en categorías ordinarias de alta rotación (como refrescos, pan y carne) y 
consistentes con líneas promocionales o de regalo, lo que genera un riesgo crítico de 
división por cero y propagación silenciosa de valores `inf`/`NaN` al calcular precios 
unitarios según la fórmula oficial (`sales_value / quantity`), requiriendo un filtrado 
explícito previo a cualquier agregación de precios o rentabilidad.

*(Ver celdas de código 5-8)*

---

**3. `transaction_data` no registra devoluciones mediante valores negativos**

Contrario a la advertencia del enunciado sobre devoluciones reflejadas con montos o 
unidades en negativo, la tabla `transaction_data` registra estrictamente 0 filas con 
valores negativos en `QUANTITY` o `SALES_VALUE` (ambas con mínimo en `0.0`); en 
consecuencia, el equipo no debe aplicar filtros de exclusión de negativos ni esperar 
neteos automáticos de devoluciones mediante signos negativos en esta tabla, debiendo 
calcular los ingresos brutos directamente sobre `SALES_VALUE` y asumiendo que el análisis 
de devoluciones no puede aislarse por esa vía en esta fuente de datos.

*(Ver celda de código 4)*

---

**Nota secundaria de calidad (Outliers en `QUANTITY` y ventas de combustible):**

Al inspeccionar los valores extremos de la variable `QUANTITY`, se identificó un máximo 
atípico de 89,638 unidades en una sola transacción. Al cruzar con la tabla `product`, 
el top 10 de cantidades más elevadas se concentra de forma sistemática en dos 
identificadores de producto (`PRODUCT_ID` 6534178 y 6544236), correspondientes a 
transacciones de quiosco y combustible (`KIOSK-GAS` / `MISC SALES TRAN`). Este patrón 
sugiere que la unidad de medida en estas transacciones podría no corresponder a 
artículos individuales de supermercado, sino a fracciones de galón, importes monetarios 
u otra unidad de venta a granel. En consecuencia, estas categorías deben aislarse o 
tratarse por separado en el pipeline de análisis para evitar distorsiones severas en el 
cálculo del tamaño promedio de canasta, elasticidad o volumen agregado.

---

**Nota adicional:** la tabla de columnas de `hh_demographic` en el User Guide (p.4) no 
coincide con las columnas reales del CSV — el documento muestra por error los nombres 
de columna de `transaction_data`. Se usaron los nombres verificados directamente con 
`.columns.tolist()`.

In [12]:
# 11. Estructura general de hh_demographic: filas, columnas, tipos
print("=== INFO DE HH_DEMOGRAPHIC ===")
hh_demographic.info(show_counts=True)

# 12. Nulos por columna
print("\n=== NULOS POR COLUMNA ===")
null_summary_hh = pd.DataFrame(
    {
        "Nulos": hh_demographic.isnull().sum(),
        "Porcentaje (%)": (
            hh_demographic.isnull().mean() * 100
        ).round(2),
    }
)
print(null_summary_hh)

=== INFO DE HH_DEMOGRAPHIC ===
<class 'pandas.DataFrame'>
RangeIndex: 801 entries, 0 to 800
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   classification_1   801 non-null    str  
 1   classification_2   801 non-null    str  
 2   classification_3   801 non-null    str  
 3   HOMEOWNER_DESC     801 non-null    str  
 4   classification_5   801 non-null    str  
 5   classification_4   801 non-null    str  
 6   KID_CATEGORY_DESC  801 non-null    str  
 7   household_key      801 non-null    int64
dtypes: int64(1), str(7)
memory usage: 50.2 KB

=== NULOS POR COLUMNA ===
                   Nulos  Porcentaje (%)
classification_1       0             0.0
classification_2       0             0.0
classification_3       0             0.0
HOMEOWNER_DESC         0             0.0
classification_5       0             0.0
classification_4       0             0.0
KID_CATEGORY_DESC      0             0.0
household_key 

In [13]:
#13. Inspeccionar valores únicos de cada columna demográfica
for col in hh_demographic.columns:
    if col != "household_key" and col != "HOUSEHOLD_KEY":
        print(f"=== {col} ===")
        print(hh_demographic[col].value_counts(dropna=False))
        print()

=== classification_1 ===
classification_1
Age Group4    288
Age Group3    194
Age Group2    142
Age Group6     72
Age Group5     59
Age Group1     46
Name: count, dtype: int64

=== classification_2 ===
classification_2
Y    344
X    340
Z    117
Name: count, dtype: int64

=== classification_3 ===
classification_3
Level5     192
Level4     172
Level6      96
Level3      77
Level2      74
Level1      61
Level8      38
Level7      34
Level9      30
Level12     11
Level10     11
Level11      5
Name: count, dtype: int64

=== HOMEOWNER_DESC ===
HOMEOWNER_DESC
Homeowner          504
Unknown            233
Renter              42
Probable Renter     11
Probable Owner      11
Name: count, dtype: int64

=== classification_5 ===
classification_5
Group5    255
Group4    187
Group3    144
Group2     95
Group6     73
Group1     47
Name: count, dtype: int64

=== classification_4 ===
classification_4
2     318
1     255
3     109
5+     66
4      53
Name: count, dtype: int64

=== KID_CATEGORY_DESC ===


In [15]:
# 14. verificación de columnas_categoricas
columnas_categoricas = hh_demographic.select_dtypes(include=["object"]).columns
print(columnas_categoricas)

Index(['classification_1', 'classification_2', 'classification_3',
       'HOMEOWNER_DESC', 'classification_5', 'classification_4',
       'KID_CATEGORY_DESC'],
      dtype='str')


C:\Users\noele\AppData\Local\Temp\ipykernel_14880\3123601662.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  columnas_categoricas = hh_demographic.select_dtypes(include=["object"]).columns


In [16]:
# 15. Detección sistemática de valores enmascarados (Unknown/None/Null)

# Seleccionar columnas de tipo texto (excluyendo identificadores numéricos)
columnas_categoricas = hh_demographic.select_dtypes(include=["str"]).columns

# Buscar patrones de datos no declarados ('Unknown', 'None', 'Null', etc.)
patron = r"Unknown|None|Null"

resumen_ocultos = []
for col in columnas_categoricas:
    # Contar coincidencias (sin distinguir mayúsculas/minúsculas)
    coincidencias = (
        hh_demographic[col]
        .astype(str)
        .str.contains(patron, case=False, na=False)
        .sum()
    )
    pct = (coincidencias / len(hh_demographic)) * 100
    resumen_ocultos.append(
        {"columna": col, "conteo_ocultos": coincidencias, "porcentaje": pct}
    )

# Presentar como DataFrame ordenado
df_ocultos = pd.DataFrame(resumen_ocultos).sort_values(
    by="conteo_ocultos", ascending=False
)
print("=== DETECCIÓN DE VALORES ENMASCARADOS / UNKNOWN ===")
print(df_ocultos.to_string(index=False))

=== DETECCIÓN DE VALORES ENMASCARADOS / UNKNOWN ===
          columna  conteo_ocultos  porcentaje
KID_CATEGORY_DESC             558   69.662921
   HOMEOWNER_DESC             233   29.088639
 classification_1               0    0.000000
 classification_3               0    0.000000
 classification_2               0    0.000000
 classification_5               0    0.000000
 classification_4               0    0.000000


### Actualización — reemplazo de afirmación #3

Tras perfilar `hh_demographic` a fondo, se identificó un hallazgo más crítico que la 
afirmación original #3. Se reemplaza:

~~**3. `transaction_data` no registra devoluciones mediante valores negativos**~~ 
→ movida a nota secundaria (ver abajo).

**Nueva afirmación #3: `KID_CATEGORY_DESC` enmascara ausencia real de hijos y no-respuesta  bajo una misma etiqueta**

La tabla `hh_demographic` presenta una tasa de datos ausentes enmascarados de 69.66% 
(558 de 801 hogares) en el campo `KID_CATEGORY_DESC` bajo el valor compuesto 
`'None/Unknown'`, así como un 29.09% (233 hogares) en `'Unknown'` para `HOMEOWNER_DESC`. 
Debido a que la etiqueta unifica en un solo registro la ausencia real de hijos con la 
no respuesta del cliente, el equipo documenta una ambigüedad semántica que este análisis 
no resuelve, desaconsejando segmentar estrategias o inferir el perfil familiar basándose 
estrictamente en esta clasificación.

*(Ver celdas de código 13-15)*

---

**Nota secundaria (antes afirmación #3): ausencia de valores negativos en `transaction_data`**

Contrario a la advertencia del enunciado sobre devoluciones reflejadas con montos o 
unidades en negativo, la tabla `transaction_data` registra estrictamente 0 filas con 
valores negativos en `QUANTITY` o `SALES_VALUE` (ambas con mínimo en `0.0`); en 
consecuencia, el equipo no debe aplicar filtros de exclusión de negativos ni esperar 
neteos automáticos de devoluciones mediante signos negativos en esta tabla, debiendo 
calcular los ingresos brutos directamente sobre `SALES_VALUE` y asumiendo que el análisis 
de devoluciones no puede aislarse por esa vía en esta fuente de datos.

*(Ver celda de código 4)*